# 第 7 章练习：把反向传播组织成可复用的层

第 6 章我们逐行写出了反向传播；这一章不增加新的数学公式，而是把相同公式封装成 `linear_forward`、`linear_backward`、`relu_forward` 和 `relu_backward`。

目标是理解：**网络变深时，变化的是层的数量，不是每层的反向传播规则。**

## 网络结构与形状

本实验仍然拟合 `y = |x|`：

```text
X (N, 1)
  ↓ linear_forward
z1 (N, 2)
  ↓ ReLU
a1 (N, 2)
  ↓ linear_forward
prediction (N, 1)
```

每个 forward 都保存 cache；对应的 backward 使用 cache 计算梯度。

In [1]:
import numpy as np

from layers import (
    linear_forward,
    linear_backward,
    relu_forward,
    relu_backward,
    mse_loss,
    mse_backward,
)

X = np.array([[-2.], [-1.], [0.], [1.], [2.]])
targets = np.array([[2.], [1.], [0.], [1.], [2.]])

W1 = np.array([[0.8, -0.6]])
b1 = np.array([0.1, 0.1])
W2 = np.array([[0.7], [0.6]])
b2 = np.array([0.1])

alpha = 0.05


In [2]:
z1, cache1 = linear_forward(X, W1, b1)
a1, cache_relu = relu_forward(z1)
prediction, cache2 = linear_forward(a1, W2, b2)

loss = mse_loss(prediction, targets)
print('z1.shape =', z1.shape)
print('a1.shape =', a1.shape)
print('prediction.shape =', prediction.shape)
print('loss =', loss)


z1.shape = (5, 2)
a1.shape = (5, 2)
prediction.shape = (5, 1)
loss = 0.21147


## 反向传播的顺序

误差从输出层向输入层反向流动：

```text
loss
 ↓ mse_backward
dz2
 ↓ linear_backward
da1, dW2, db2
 ↓ relu_backward
dz1
 ↓ linear_backward
dX, dW1, db1
```

`dW` 和 `db` 更新当前层；`dX` 只是把误差信号继续传给前一层。

In [3]:
dz2 = mse_backward(prediction, targets)

da1, dW2, db2 = linear_backward(dz2, cache2)
dz1 = relu_backward(da1, cache_relu)
dX, dW1, db1 = linear_backward(dz1, cache1)

print('dW1.shape =', dW1.shape)
print('db1.shape =', db1.shape)
print('dW2.shape =', dW2.shape)
print('db2.shape =', db2.shape)


dW1.shape = (1, 2)
db1.shape = (2,)
dW2.shape = (2, 1)
db2.shape = (1,)


In [4]:
for epoch in range(2000):
    z1, cache1 = linear_forward(X, W1, b1)
    a1, cache_relu = relu_forward(z1)
    prediction, cache2 = linear_forward(a1, W2, b2)

    loss = mse_loss(prediction, targets)
    dz2 = mse_backward(prediction, targets)
    da1, dW2, db2 = linear_backward(dz2, cache2)
    dz1 = relu_backward(da1, cache_relu)
    dX, dW1, db1 = linear_backward(dz1, cache1)

    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2

    if epoch % 400 == 0:
        print(epoch, np.round(prediction.ravel(), 4), loss)

print('target =', targets.ravel())
print('prediction =', np.round(prediction.ravel(), 4))


0 [0.88 0.52 0.23 0.73 1.29] 0.21147
400 [1.9978 1.0021 0.0067 1.002  1.9979] 6.278565895273289e-06
800 [2.000e+00 1.000e+00 2.000e-04 9.999e-01 2.000e+00] 4.456222450971854e-09
1200 [2.000e+00 1.000e+00 1.000e-04 9.999e-01 2.000e+00] 9.040280141305331e-10
1600 [2. 1. 0. 1. 2.] 3.3697062705375067e-10
target = [2. 1. 0. 1. 2.]
prediction = [2. 1. 0. 1. 2.]


## 小结

这次没有新增求导公式，只是把已有公式组织起来：

- forward 保存中间结果；
- backward 使用 cache 计算梯度；
- 参数梯度更新当前层；
- 输入梯度传给前一层。

下一步再增加一层时，只需要重复这个接口，而不是重新写一套特殊代码。